# Evaluate Classifier

This notebook evaluates the trained dice classifier on the validation split.

Goals:
- compute overall accuracy
- inspect per-class performance
- visualize the confusion matrix
- display misclassified examples

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

from project_config import CLASSIFICATION_DIR, MODELS_DIR, CLASSIFIER_IMG_SIZE, CLASSIFIER_BATCH_SIZE

In [ ]:
VAL_DIR = CLASSIFICATION_DIR / "val"
MODEL_PATH = MODELS_DIR / "dice_classifier_best.keras"
MAX_SHOW_ERRORS = 18

assert VAL_DIR.exists(), f"Missing val dir: {VAL_DIR}"
assert MODEL_PATH.exists(), f"Missing model: {MODEL_PATH}"

print("VAL_DIR:", VAL_DIR)
print("MODEL_PATH:", MODEL_PATH)

In [ ]:
val_ds_raw = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    labels="inferred",
    label_mode="int",
    image_size=CLASSIFIER_IMG_SIZE,
    batch_size=CLASSIFIER_BATCH_SIZE,
    shuffle=False,
)

class_names = val_ds_raw.class_names
num_classes = len(class_names)

def preprocess_eval(images, labels):
    images = tf.cast(images, tf.float32)
    images = keras.applications.efficientnet.preprocess_input(images)
    return images, labels

val_ds = val_ds_raw.map(preprocess_eval, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

model = keras.models.load_model(MODEL_PATH)

In [ ]:
all_images = []
y_true = []

for images_raw, labels_raw in val_ds_raw:
    all_images.append(images_raw.numpy())
    y_true.extend(labels_raw.numpy().tolist())

all_images = np.concatenate(all_images, axis=0)
y_true = np.array(y_true, dtype=np.int32)

pred_probs = model.predict(val_ds, verbose=1)
y_pred = np.argmax(pred_probs, axis=1)
y_conf = np.max(pred_probs, axis=1)

assert len(all_images) == len(y_true) == len(y_pred)

In [ ]:
accuracy = float(np.mean(y_true == y_pred))

print("\nCOPY_PASTE_CLASSIFIER_SUMMARY_START")
print(f"samples={len(y_true)}")
print(f"accuracy={accuracy:.4f}")
print("COPY_PASTE_CLASSIFIER_SUMMARY_END")

In [ ]:
cm = np.zeros((num_classes, num_classes), dtype=np.int32)
for t, p in zip(y_true, y_pred):
    cm[t, p] += 1

print("\nConfusion matrix (rows=true, cols=pred):")
print(cm)

print("\nCOPY_PASTE_PER_CLASS_START")
for i, cls_name in enumerate(class_names):
    total = int(np.sum(cm[i]))
    correct = int(cm[i, i])
    cls_acc = correct / total if total > 0 else 0.0
    print(f"class={cls_name} total={total} correct={correct} accuracy={cls_acc:.4f}")
print("COPY_PASTE_PER_CLASS_END")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(cm)

ax.set_xticks(np.arange(num_classes))
ax.set_yticks(np.arange(num_classes))
ax.set_xticklabels(class_names)
ax.set_yticklabels(class_names)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Classifier confusion matrix")

for i in range(num_classes):
    for j in range(num_classes):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center")

plt.tight_layout()
plt.show()

In [ ]:
wrong_idx = np.where(y_true != y_pred)[0]
print(f"\nMisclassified samples: {len(wrong_idx)} / {len(y_true)}")

if len(wrong_idx) > 0:
    show_idx = wrong_idx[:MAX_SHOW_ERRORS]
    cols = 3
    rows = int(np.ceil(len(show_idx) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = np.array(axes).reshape(-1)

    for ax in axes[len(show_idx):]:
        ax.axis("off")

    for plot_i, sample_i in enumerate(show_idx):
        ax = axes[plot_i]
        image = all_images[sample_i].astype(np.uint8)
        true_label = class_names[y_true[sample_i]]
        pred_label = class_names[y_pred[sample_i]]
        conf = float(y_conf[sample_i])

        ax.imshow(image)
        ax.set_title(f"T:{true_label}  P:{pred_label}  ({conf:.2f})")
        ax.axis("off")

    plt.tight_layout()
    plt.show()